In [2]:
%matplotlib inline
from pathlib import Path
import sys

PROJECT_ROOT = Path("/data/dn/FRTP_revision1")
IMAGECLS_ROOT = PROJECT_ROOT / "imagecls"
if str(IMAGECLS_ROOT) not in sys.path:
    sys.path.insert(0, str(IMAGECLS_ROOT))

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from custom_datasets import _make_loader, make_label_drop_subset, make_recovered_target_replay_subset
from FRPT import get_score, save_recons_fea_to_h5

DEVICE = torch.device("cuda:6" if torch.cuda.is_available() else "cpu")
DATA_ROOT = PROJECT_ROOT / "mydata"
CKPT_ROOT = IMAGECLS_ROOT / "ckpts" / "nette"
print("DEVICE:", DEVICE)


DEVICE: cuda:6


In [3]:
"""数据"""
# Imagenette/Imagewoof are ImageNet subsets, so use the standard ImageNet normalization.
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

transform_train = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.65, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandAugment(num_ops=2, magnitude=7),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    transforms.RandomErasing(p=0.15),
])
transform_test = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

DATASET_NAME = "nette"
DATASET_DIR = DATA_ROOT / "imagenette2"
trainset = datasets.ImageFolder(str(DATASET_DIR / "train"), transform=transform_train)
testset = datasets.ImageFolder(str(DATASET_DIR / "val"), transform=transform_test)
print(len(trainset), len(testset))
BATCH_SIZE = 1024
NUM_WORKERS = 4

train_loader = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True) 
test_loader = DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False) 


9469 3925


In [6]:
def train_base(model,trainloader,testloader, optimizer, num_epochs, es_patience=5,scheduler=None,on_train_epoch_start=None,stage_name="train",):
    # if scheduler is None:
    #     scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    #         optimizer, mode='max', factor=0.5, patience=3, threshold=0.0, min_lr=1e-6
    #     )
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

    test_acc_ls, loss_ls, train_acc_ls = [], [], []
    best_acc, best_ckpt, wait = -1.0, None, 0

    for epoch in range(num_epochs):
        model.train()
        if on_train_epoch_start is not None:
            on_train_epoch_start(model)
        epoch_loss, sample_num = 0.0, 0

        for x, y in trainloader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            loss = criterion(model(x)['out'], y)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

            bs = x.size(0)
            epoch_loss += loss.item() * bs
            sample_num += bs

        epoch_loss /= sample_num
        loss_ls.append(epoch_loss)

        test_acc = eval_acc(model, testloader)
        test_acc_ls.append(test_acc)
        lr_desc = ", ".join(f"{group['lr']:.2e}" for group in optimizer.param_groups)
        
        train_acc_ls.append(eval_acc(model, trainloader)) # use?

        if len(train_acc_ls):
            print(f'[{stage_name}] Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.6f}, Train Accuracy: {train_acc_ls[-1]:.4f}, Test Accuracy: {test_acc:.4f}, LR: {lr_desc}')
        else:
            print(f'[{stage_name}] Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.6f}, Test Accuracy: {test_acc:.4f}, LR: {lr_desc}')
        
        if ((epoch + 1) % 5 == 0 and (epoch + 1) not in []) or (epoch + 1) == num_epochs:
            periodic_ckpt = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            name = getattr(model, 'name', model.__class__.__name__)
            fname = CKPT_ROOT / f"{name}_{PRETRAIN_TAG}_e{epoch+1}_{test_acc:.4f}.pth"
            try:
                torch.save(periodic_ckpt, str(fname))
                print(f"Saved periodic checkpoint to {fname}")
            except Exception as e:
                print(f"Failed saving periodic checkpoint: {e}")

        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(test_acc)
            else:
                scheduler.step()
            if test_acc > best_acc:
                best_acc = test_acc
                best_ckpt = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                wait = 0
            else:
                wait += 1
                if es_patience is not None and wait >= es_patience:
                    print(f'[{stage_name}] Early stopping at epoch {epoch+1}. Best Test Accuracy: {best_acc:.4f}')
                    break

    if scheduler is not None:
        model.load_state_dict(best_ckpt)
    return loss_ls, train_acc_ls, test_acc_ls, best_ckpt


def _set_trainable(module, trainable):
    for p in module.parameters():
        p.requires_grad = trainable


def _count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def _split_backbone_head_params(model):
    head_params = list(model.fc.parameters())
    head_ids = {id(p) for p in head_params}
    backbone_params = [p for p in model.parameters() if id(p) not in head_ids]
    return backbone_params, head_params


def train_twostage(
    model, trainloader, testloader,head_lr=1e-3,backbone_lr=2e-5,finetune_head_lr=2e-4, head_epochs=5,finetune_epochs=30, head_patience=3,
    finetune_patience=None, weight_decay=1e-4, label_smoothing=0.05):
    """
    Two-stage finetuning for pretrained ResNet18/34:
    1. Freeze backbone and train classifier head.
    2. Unfreeze all parameters with smaller backbone lr and larger head lr.
    """
    if not hasattr(model, 'fc'):
        raise AttributeError("train_twostage expects a ResNet-style model with model.fc as classifier head")

    head = model.fc
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    _set_trainable(model, False)
    _set_trainable(head, True)

    def keep_backbone_eval_head_train(m):
        # train_base calls model.train() each epoch; reset frozen BN/dropout behavior here.
        m.eval()
        head.train()

    print(f"Stage 1/2: train classifier head only, lr={head_lr:.2e}, epochs={head_epochs}, trainable_params={_count_trainable_params(model):,}")
    head_optimizer = torch.optim.AdamW(head.parameters(), lr=head_lr, weight_decay=weight_decay)
    head_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        head_optimizer, T_max=head_epochs, eta_min=1e-5
    )
    head_loss, head_trainacc, head_testacc, head_best_ckpt = train_base(
        model,
        trainloader,
        testloader,
        head_optimizer,
        head_epochs,
        es_patience=head_patience,
        scheduler=head_scheduler,
        # criterion=criterion,
        on_train_epoch_start=keep_backbone_eval_head_train,
        stage_name="head",
    )

    model.load_state_dict(head_best_ckpt)
    _set_trainable(model, True)
    backbone_params, head_params = _split_backbone_head_params(model)

    print(
        f"Stage 2/2: finetune all parameters, backbone_lr={backbone_lr:.2e}, "
        f"head_lr={finetune_head_lr:.2e}, epochs={finetune_epochs}, "
        f"trainable_params={_count_trainable_params(model):,}"
    )
    finetune_optimizer = torch.optim.AdamW(
        [
            {"params": backbone_params, "lr": backbone_lr},
            {"params": head_params, "lr": finetune_head_lr},
        ],
        weight_decay=weight_decay,
    )
    finetune_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        finetune_optimizer, T_max=finetune_epochs, eta_min=1e-6
    )
    finetune_loss, finetune_trainacc, finetune_testacc, finetune_best_ckpt = train_base(
        model,
        trainloader,
        testloader,
        finetune_optimizer,
        finetune_epochs,
        es_patience=finetune_patience,
        scheduler=finetune_scheduler,
        # criterion=criterion,
        stage_name="finetune",
    )

    loss_ls = head_loss + finetune_loss
    train_acc_ls = head_trainacc + finetune_trainacc
    test_acc_ls = head_testacc + finetune_testacc
    best_ckpt = finetune_best_ckpt if max(finetune_testacc) >= max(head_testacc) else head_best_ckpt
    model.load_state_dict(best_ckpt)
    return loss_ls, train_acc_ls, test_acc_ls, best_ckpt


## baseline


In [7]:
"""train baseline model: pretrained ResNet34, head training then discriminative finetuning"""
from models import SimpleCNN_nette, ResNet_nette
# model = SimpleCNN_nette(activate=torch.relu, version="v2").to(DEVICE)
# model.load_state_dict(torch.load("/data/dn/FRTP_revision1/imagecls/ckpts_ood/nette/nette_simplecnnv2_relu_target0_n01440764_drop0.9_e10_0.5327.pth", map_location=DEVICE))
# optimizer_ = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
# EPOCHES = 10
# loss_lt, trainacc_lt, testacc_lt, best_ckpt = train_base(model,train_loader,test_loader, optimizer_,EPOCHES, scheduler=None)

# model = ResNet_nette(version='18', pretrain=True).to(DEVICE)
# print(model.name, model.pretrain_info)
# loss_lt, trainacc_lt, testacc_lt, best_ckpt = train_twostage(
#     model, train_loader, test_loader, head_lr=1e-3,backbone_lr=2e-5,
#     finetune_head_lr=2e-4, head_epochs=5, finetune_epochs=30,head_patience=3,finetune_patience=None,
#     weight_decay=1e-4, label_smoothing=0.05,)

model = ResNet_nette(version='18', pretrain=False).to(DEVICE)
model.load_state_dict(torch.load("/data/dn/FRTP_revision1/imagecls/ckpts_ood/nette/nette_resnet18_target0_n01440764_drop0.9_e80_0.8522.pth", map_location=DEVICE))
optimizer_ = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
EPOCHES = 20
loss_lt, trainacc_lt, testacc_lt, best_ckpt = train_base(model,train_loader,test_loader, optimizer_,EPOCHES, scheduler=None)

print("baseline_loss.extend(")
print(loss_lt)
print("baseline_testacc.extend(")
print(testacc_lt)

/home/dn/.tmp/ipykernel_327935/893279799.py:17: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("/data/dn/FRTP_revision1/imagecls/ckpts_ood/ne

FileNotFoundError: [Errno 2] No such file or directory: '/data/dn/FRTP_revision1/imagecls/ckpts_ood/nette/nette_resnet18_target0_n01440764_drop0.9_e80_0.8522.pth'

## get recons data


In [4]:
from models import SimpleCNN_nette, ResNet_nette
# model = ResNet_nette(version='18', pretrain=False).to(DEVICE)
# MODELPATH = "/data/dn/FRTP_revision1/imagecls/ckpts/nette/nette_resnet18_epoch5_0.9814.pth"
model = SimpleCNN_nette(activate=torch.relu, version="v2").to(DEVICE)
MODELPATH = "/data/dn/FRTP_revision1/imagecls/ckpts/nette/nette_simplecnnv2_relu_epoch10_0.5934.pth"
ckpt = torch.load(MODELPATH, map_location=DEVICE)
model.load_state_dict(ckpt)
model.eval()
# test_score = get_score(model, test_loader, DEVICE)
# TESTACC = test_score * 100
# train_score = get_score(model, train_loader, DEVICE)
# TRAINACC = train_score * 100
# print(f"testacc={TESTACC:.2f}%, trainacc={TRAINACC:.2f}%")
model.check_cond()


/home/dn/.tmp/ipykernel_641226/807365439.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(MODELPATH, map_location=DEVICE)


conv1.weight torch.Size([16, 3, 25, 25]) --------------------
tensor([[25, 25, 25],
        [25, 25, 25],
        [25, 25, 25],
        [25, 25, 25],
        [25, 25, 25],
        [25, 25, 25],
        [25, 25, 25],
        [25, 25, 25],
        [25, 25, 25],
        [25, 25, 25],
        [25, 25, 25],
        [25, 25, 25],
        [25, 25, 25],
        [25, 25, 25],
        [25, 25, 25],
        [25, 25, 25]], device='cuda:6')
tensor([[ 653.7025,   78.3972,  240.1993],
        [ 358.9546,  235.5022,  321.7914],
        [ 924.2406,  153.6571,  191.4032],
        [ 300.8494,  110.2217,  121.2260],
        [ 709.2466,  432.8712, 2157.8730],
        [ 130.5562,  951.3842,  437.1841],
        [ 679.4562,  113.3112, 4548.6094],
        [ 239.1747,  696.2939,  801.7161],
        [ 272.5975, 1566.8801,  110.8068],
        [ 237.6427,  136.7631,  363.5607],
        [ 186.2836,  528.8867,  220.1817],
        [ 711.1118, 5382.0586,  857.6725],
        [  93.5518,  200.0649, 6334.9233],
        [

In [ ]:
FILEPATH = Path("/data/dn/FRTP_revision1/imagecls/recons_data_new/nette") / f"{Path(MODELPATH).stem}.h5"
save_recons_fea_to_h5(model, train_loader, str(FILEPATH), DEVICE)

saved recons data to /data/dn/FRTP_revision1/imagecls/recons_data_new/nette/nette_simplecnnv2_relu_epoch10_0.5934.h5


: 

## FRPT


In [ ]:
from models import SimpleCNN_nette, ResNet_nette
# model = SimpleCNN_nette(activate=torch.relu, version='v2').to(DEVICE)
# MODELPATH = "/data/dn/FRTP_revision1/imagecls/ckpts/nette/nette_simplecnnv2_relu_0.4897.pth"
model = ResNet_nette(version='18', pretrain=False).to(DEVICE)
MODELPATH = "/data/dn/FRTP_revision1/imagecls/ckpts/nette/nette_resnet18_epoch5_0.9814.pth"
EPOCHS = 10
SEED_ls = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9][:1]
ALPHA_ls = [0, 0.1]

summary_res = model_post_train_all(model, MODELPATH, test_loader,
                            ALPHA_ls, SEED_ls, EPOCHS, DEVICE, BATCH_SIZE, save_res=False)


[1] testacc=0.978854, loss=0.130912, reconsloss=2.084435, clsloss=0.130912, lossweight=0.000000
[2] testacc=0.980892, loss=0.016046, reconsloss=2.721993, clsloss=0.016046, lossweight=0.000000
[3] testacc=0.981401, loss=0.005247, reconsloss=3.205923, clsloss=0.005247, lossweight=0.000000
[4] testacc=0.982166, loss=0.002215, reconsloss=3.696716, clsloss=0.002215, lossweight=0.000000
[5] testacc=0.980637, loss=0.001787, reconsloss=4.078947, clsloss=0.001787, lossweight=0.000000
[6] testacc=0.980892, loss=0.001299, reconsloss=4.532112, clsloss=0.001299, lossweight=0.000000
[7] testacc=0.981401, loss=0.001130, reconsloss=4.533885, clsloss=0.001130, lossweight=0.000000
[8] testacc=0.980892, loss=0.000789, reconsloss=4.913658, clsloss=0.000789, lossweight=0.000000
[9] testacc=0.980637, loss=0.000908, reconsloss=4.898341, clsloss=0.000908, lossweight=0.000000
[10] testacc=0.981146, loss=0.000721, reconsloss=5.173280, clsloss=0.000721, lossweight=0.000000
[RECORD] alpha=0, recons_key=recons_out

## ablation


In [ ]:
from models import SimpleCNN_nette
model = SimpleCNN_nette(activate=torch.relu, version="v1").to(DEVICE)
MODELPATH = "/data/dn/FRTP_revision1/imagecls/ckpts/nette/nette_simplecnnv1_relu_0.4339.pth"
EPOCHS = 10
ALPHA_ls = [0, 0.1, 0.3, 0.5, 0.7, 0.9]
SEED_ls = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9][:1]

ablation_summary = ablation_all(
    model, MODELPATH, test_loader, ALPHA_ls, SEED_ls, EPOCHS, DEVICE, BATCH_SIZE, save_res=False)

## visualize


In [ ]:
from pathlib import Path
from matplotlib import colormaps
from matplotlib.colors import BoundaryNorm
from models import SimpleCNN_nette

DATASET_NAME = globals().get("DATASET_NAME", "nette")
PROJECT_ROOT = Path("/data/dn/FRTP_revision1")
CKPT_ROOT = PROJECT_ROOT / "imagecls" / "ckpts"
VIS_MEAN = torch.tensor((0.485, 0.456, 0.406)).view(3, 1, 1)
VIS_STD = torch.tensor((0.229, 0.224, 0.225)).view(3, 1, 1)
VIS_FEATURE_NAME = "z1"
VIS_SAMPLE_IDX = 0
VIS_TARGET_LABEL = None
VIS_MAX_CHANNELS = 6
VIS_ALPHA = 0.70
VIS_DIFF_PERCENTILE = 0.90
VIS_HEATMAP_LEVELS = 7
VIS_CONV_METHOD = "fft_pad"

vis_model = SimpleCNN_nette(activate=torch.relu, version="v1").to(DEVICE)
if "MODELPATH" not in globals():
    ckpt_candidates = sorted((CKPT_ROOT / DATASET_NAME).glob(f"{vis_model.name}_*.pth"))
    assert ckpt_candidates, f"No checkpoint found for {vis_model.name}; run baseline cell or set MODELPATH manually."
    MODELPATH = ckpt_candidates[-1]
MODELPATH = Path(MODELPATH)
vis_model.load_state_dict(torch.load(MODELPATH, map_location=DEVICE, weights_only=True))
vis_model.eval()

def denormalize_img(x):
    x = x.detach().cpu().squeeze(0).float()
    if x.dim() == 2:
        x = x.unsqueeze(0)
    if x.shape[0] == 3:
        x = x * VIS_STD + VIS_MEAN
    x = x.clamp(0, 1)
    return x.permute(1, 2, 0).numpy() if x.shape[0] == 3 else x.squeeze(0).numpy()

def normalize_map(x):
    if isinstance(x, torch.Tensor):
        x = x.detach().cpu().float()
    x_min, x_max = x.min(), x.max()
    return (x - x_min) / (x_max - x_min + 1e-12)

def pick_visual_sample(model, dataset, sample_idx=0, target_label=None, device=DEVICE):
    if target_label is None:
        x, y = dataset[sample_idx]
        x_dev = x.unsqueeze(0).to(device)
        with torch.no_grad():
            logits = model(x_dev)["out"].detach().cpu()
        pred = int(logits.argmax(dim=1).item())
        return sample_idx, x_dev, int(y), pred, logits

    loader = DataLoader(dataset, batch_size=1, shuffle=False)
    model.eval()
    with torch.no_grad():
        for idx, (x, y) in enumerate(loader):
            if int(y.item()) != target_label:
                continue
            x_dev = x.to(device)
            logits = model(x_dev)["out"].detach().cpu()
            pred = int(logits.argmax(dim=1).item())
            return idx, x_dev, int(y.item()), pred, logits
    raise RuntimeError(f"No sample found for target_label={target_label}")

def visualize_forward_recons_diff(model, dataset, feature_name="z1", sample_idx=0, target_label=None):
    idx, input_data, label, pred, logits = pick_visual_sample(model, dataset, sample_idx, target_label)
    with torch.no_grad():
        forward_res = model(input_data)
        recons_res = model.get_recons_fea(input_data, torch.tensor([label], device=input_data.device), conv_method=VIS_CONV_METHOD)

    if feature_name == "out":
        forward_feature = forward_res["out"].detach().cpu().flatten()
        recons_feature = recons_res["recons_out"].detach().cpu().flatten()
        diff = recons_feature - forward_feature
        fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
        axes[0].imshow(denormalize_img(input_data))
        axes[0].set_title(f"idx={idx}, gt={label}, pred={pred}")
        axes[0].axis("off")
        axes[1].plot(forward_feature.numpy(), label="forward")
        axes[1].plot(recons_feature.numpy(), label="recons")
        axes[1].plot(diff.numpy(), label="diff")
        axes[1].legend()
        axes[1].set_title("out / recons_out")
        plt.show()
        return

    forward_feature = forward_res[feature_name].detach().cpu().squeeze(0)
    recons_feature = recons_res[f"recons_{feature_name}"].detach().cpu().squeeze(0)
    assert forward_feature.shape == recons_feature.shape, (forward_feature.shape, recons_feature.shape)
    diff = recons_feature - forward_feature
    diff_quantile = VIS_DIFF_PERCENTILE / 100 if VIS_DIFF_PERCENTILE > 1 else VIS_DIFF_PERCENTILE
    vmax = torch.quantile(diff.abs().flatten(), diff_quantile).item()
    vmax = max(vmax, diff.abs().max().item() * 1e-6, 1e-12)
    channel_count = min(forward_feature.size(0), VIS_MAX_CHANNELS)
    bounds = np.linspace(-vmax, vmax, VIS_HEATMAP_LEVELS + 1)
    cmap = colormaps["bwr"].resampled(VIS_HEATMAP_LEVELS)
    norm = BoundaryNorm(bounds, cmap.N, clip=True)

    fig, axes = plt.subplots(3, channel_count + 1, figsize=(3.0 * (channel_count + 1), 8.5), squeeze=False)
    img = denormalize_img(input_data)
    for r, title in enumerate(["input / forward", "recons", "forward + diff"]):
        axes[r, 0].imshow(img)
        axes[r, 0].set_title(title if r else f"idx={idx}\ngt={label}, pred={pred}")
        axes[r, 0].axis("off")

    im = None
    for c in range(channel_count):
        f_base = normalize_map(forward_feature[c]).numpy()
        r_base = normalize_map(recons_feature[c]).numpy()
        d = diff[c].numpy()
        axes[0, c + 1].imshow(f_base, cmap="gray")
        axes[0, c + 1].set_title(f"forward {feature_name} ch{c}")
        axes[1, c + 1].imshow(r_base, cmap="gray")
        axes[1, c + 1].set_title(f"recons {feature_name} ch{c}")
        axes[2, c + 1].imshow(f_base, cmap="gray")
        im = axes[2, c + 1].imshow(d, cmap=cmap, norm=norm, alpha=VIS_ALPHA)
        axes[2, c + 1].set_title(f"diff ch{c}")
        for r in range(3):
            axes[r, c + 1].axis("off")

    fig.colorbar(im, ax=axes[:, 1:].ravel().tolist(), fraction=0.025, pad=0.02).set_label("recons - forward")
    fig.suptitle(f"{DATASET_NAME}: forward / reconstruction feature comparison ({MODELPATH.name})")
    plt.show()
    print(f"sample_idx={idx}, gt={label}, pred={pred}, logits_head={logits.numpy().round(3).tolist()[0][:10]}")
    print(f"diff mean={diff.mean().item():.6f}, min={diff.min().item():.6f}, max={diff.max().item():.6f}")

visualize_forward_recons_diff(vis_model, testset, VIS_FEATURE_NAME, VIS_SAMPLE_IDX, VIS_TARGET_LABEL)


In [ ]:
import torch.nn.functional as F

CAM_ALPHA = 0.55
CAM_CMAP = "jet"
CAM_DIFF_FEATURE_NAME = "out"      # 可选: "out" 或 model.get_fea_name() 对应的 z 特征，如 "z1", "z2"
CAM_RECONS_FEATURE_NAME = "out"
CAM_FORWARD_FEATURE_NAME = "out"
CAM_SAMPLE_IDX = VIS_SAMPLE_IDX if "VIS_SAMPLE_IDX" in globals() else 0
CAM_TARGET_LABEL = VIS_TARGET_LABEL if "VIS_TARGET_LABEL" in globals() else None

def tensor_to_numpy_img(x):
    return denormalize_img(x)

def upsample_cam(cam, input_data):
    cam = F.interpolate(cam, size=input_data.shape[-2:], mode="bilinear", align_corners=False)
    cam = cam.squeeze().detach().cpu().float()
    return normalize_map(cam).numpy()

def get_diff_driven_cam(model, input_data, label, feature_name="z1"):
    model.eval()
    recons_key = f"recons_{feature_name}"
    if feature_name == "out":
        x = input_data.detach().clone().requires_grad_(True)
        model.zero_grad(set_to_none=True)
        forward_out = model(x)["out"]
        recons_out = model.get_recons_fea(x, torch.tensor([label], device=x.device), conv_method=VIS_CONV_METHOD)["recons_out"]
        diff = recons_out - forward_out
        score = diff[0, label]
        score.backward()
        saliency = x.grad.detach().abs().amax(dim=1, keepdim=True)
        return upsample_cam(saliency, input_data), diff.detach().cpu()

    with torch.no_grad():
        forward_res = model(input_data)
        recons_res = model.get_recons_fea(input_data, torch.tensor([label], device=input_data.device), conv_method=VIS_CONV_METHOD)
    forward_feature = forward_res[feature_name]
    recons_feature = recons_res[recons_key]
    diff = recons_feature - forward_feature
    if diff.dim() != 4:
        raise ValueError(f"diff for {feature_name} has shape {tuple(diff.shape)}; expected 4D or feature_name='out'.")
    weights = diff.mean(dim=(2, 3), keepdim=True)
    cam_signed = (weights * forward_feature).sum(dim=1, keepdim=True)
    cam = F.relu(cam_signed)
    if cam.max().item() <= 1e-12:
        cam = cam_signed.abs()
    return upsample_cam(cam, input_data), diff.detach().cpu()

def feature_map_to_cam(feature, input_data):
    weights = feature.mean(dim=(2, 3), keepdim=True)
    cam_signed = (weights * feature).sum(dim=1, keepdim=True)
    cam = F.relu(cam_signed)
    if cam.max().item() <= 1e-12:
        cam = cam_signed.abs()
    return upsample_cam(cam, input_data)

def get_recons_feature_cam(model, input_data, label, feature_name="z1"):
    if feature_name == "out":
        x = input_data.detach().clone().requires_grad_(True)
        model.zero_grad(set_to_none=True)
        recons_out = model.get_recons_fea(x, torch.tensor([label], device=x.device), conv_method=VIS_CONV_METHOD)["recons_out"]
        score = recons_out[0, label]
        score.backward()
        saliency = x.grad.detach().abs().amax(dim=1, keepdim=True)
        return upsample_cam(saliency, input_data)
    with torch.no_grad():
        recons_feature = model.get_recons_fea(input_data, torch.tensor([label], device=input_data.device), conv_method=VIS_CONV_METHOD)[f"recons_{feature_name}"]
    if recons_feature.dim() != 4:
        raise ValueError(f"recons_{feature_name} has shape {tuple(recons_feature.shape)}; expected 4D or feature_name='out'.")
    return feature_map_to_cam(recons_feature, input_data)

def get_forward_feature_cam(model, input_data, feature_name="z1", class_idx=None):
    if feature_name == "out":
        if class_idx is None:
            with torch.no_grad():
                class_idx = int(model(input_data)["out"].argmax(dim=1).item())
        x = input_data.detach().clone().requires_grad_(True)
        model.zero_grad(set_to_none=True)
        score = model(x)["out"][0, class_idx]
        score.backward()
        saliency = x.grad.detach().abs().amax(dim=1, keepdim=True)
        return upsample_cam(saliency, input_data)
    with torch.no_grad():
        forward_feature = model(input_data)[feature_name]
    if forward_feature.dim() != 4:
        raise ValueError(f"forward feature {feature_name} has shape {tuple(forward_feature.shape)}; expected 4D or feature_name='out'.")
    return feature_map_to_cam(forward_feature, input_data)

def visualize_input_cam_compare(model, dataset, sample_idx=0, target_label=None, diff_feature_name="out", recons_feature_name="out", forward_feature_name="out"):
    idx, input_data, label, pred, logits = pick_visual_sample(model, dataset, sample_idx, target_label)
    diff_cam, diff = get_diff_driven_cam(model, input_data, label, diff_feature_name)
    recons_cam = get_recons_feature_cam(model, input_data, label, recons_feature_name)
    forward_cam = get_forward_feature_cam(model, input_data, forward_feature_name, class_idx=pred)
    input_img = tensor_to_numpy_img(input_data)

    fig, axes = plt.subplots(1, 4, figsize=(14.0, 3.6), squeeze=False)
    axes[0, 0].imshow(input_img)
    axes[0, 0].set_title(f"input\nidx={idx}, gt={label}, pred={pred}")
    axes[0, 0].axis("off")
    for ax, cam, title in [
        (axes[0, 1], diff_cam, f"diff-driven\nfeature={diff_feature_name}"),
        (axes[0, 2], recons_cam, f"recons-feature\nfeature={recons_feature_name}"),
        (axes[0, 3], forward_cam, f"forward-feature\nfeature={forward_feature_name}"),
    ]:
        ax.imshow(input_img)
        im = ax.imshow(cam, cmap=CAM_CMAP, alpha=CAM_ALPHA, vmin=0, vmax=1)
        ax.set_title(title)
        ax.axis("off")
    fig.colorbar(im, ax=axes.ravel().tolist(), fraction=0.025, pad=0.02)
    fig.suptitle(f"{DATASET_NAME} input-space highlight comparison ({MODELPATH.name})")
    plt.show()
    print(f"sample_idx={idx}, gt={label}, pred={pred}, logits_head={logits.numpy().round(3).tolist()[0][:10]}")
    print(f"diff mean={diff.mean().item():.6f}, min={diff.min().item():.6f}, max={diff.max().item():.6f}")

visualize_input_cam_compare(
    vis_model, testset, CAM_SAMPLE_IDX, CAM_TARGET_LABEL,
    CAM_DIFF_FEATURE_NAME, CAM_RECONS_FEATURE_NAME, CAM_FORWARD_FEATURE_NAME
)


## different stage vs recons loss

In [ ]:
from models import SimpleCNN_nette, ResNet_nette
train_loader = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False)
print(len(train_loader), len(test_loader))

STAGE_RECONS_MODEL_BUILDERS = {
    "simplecnnv1": lambda: SimpleCNN_nette(activate=torch.relu, version="v1"),
    "simplecnnv2": lambda: SimpleCNN_nette(activate=torch.relu, version="v2"),
    "resnet18": lambda: ResNet_nette(version="18", pretrain=False),
    "resnet34": lambda: ResNet_nette(version="34", pretrain=False),
}

def get_score(model, data_loader, DEVICE=DEVICE):
    model.eval()
    correct = 0.0
    with torch.no_grad():
        for data, target in data_loader:
            data = data.to(DEVICE, non_blocking=True)
            target = target.to(DEVICE, non_blocking=True)
            output = model(data)["out"]
            pred = output.argmax(dim=1)
            correct += pred.eq(target.view_as(pred)).sum().item()
    return correct / len(data_loader.dataset)


def get_recons_loss(model, data_loader, DEVICE=DEVICE):
    """Return the sample-averaged reconstruction loss on data_loader."""
    model.eval()
    sample_num = 0
    recons_loss = {recons_key: 0.0 for recons_key in model.get_fea_name()}
    with torch.no_grad():
        for input, target in data_loader:
            input = input.to(DEVICE, non_blocking=True)
            target = target.to(DEVICE, non_blocking=True)
            recons_feature = model.get_recons_fea(input.detach(), target, recons_key=None)
            res = model(input)
            bs = input.size(0)
            sample_num += bs
            for recons_key in model.get_fea_name():
                diff = res[recons_key[7:]] - recons_feature[recons_key]
                reconsloss_batch = diff.flatten(1).pow(2).sum(dim=1).sum()
                recons_loss[recons_key] += reconsloss_batch.item()
    return {recons_key: loss / sample_num for recons_key, loss in recons_loss.items()}


def train_stage_recons(model, lr, num_epochs, train_record=True, every_epoch=5):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    clscrit = nn.CrossEntropyLoss()
    recons_keys = model.get_fea_name()

    train_score_list, test_score_list = [], []
    train_recons_loss_list = {recons_key: [] for recons_key in recons_keys}
    test_recons_loss_list = {recons_key: [] for recons_key in recons_keys}

    for epoch in range(num_epochs):
        if epoch % every_epoch == 0:
            print("************* ", end="")
            test_score = get_score(model, test_loader, DEVICE)
            test_score_list.append(test_score)
            test_recons_loss = get_recons_loss(model, test_loader, DEVICE)
            for recons_key in recons_keys:
                test_recons_loss_list[recons_key].append(test_recons_loss[recons_key])

            if train_record:
                train_score = get_score(model, train_loader, DEVICE)
                train_score_list.append(train_score)
                train_recons_loss = get_recons_loss(model, train_loader, DEVICE)
                for recons_key in recons_keys:
                    train_recons_loss_list[recons_key].append(train_recons_loss[recons_key])
                print(
                    f"Epoch [{epoch + 1}/{num_epochs}] "
                    f"train_score={train_score:.4f}, test_score={test_score:.4f}, "
                    f"train_recons_loss={train_recons_loss}, test_recons_loss={test_recons_loss}"
                )
            else:
                print(
                    f"Epoch [{epoch + 1}/{num_epochs}] test_score={test_score:.4f}, "
                    f"test_recons_loss={test_recons_loss}"
                )

        model.train()
        for data, target in train_loader:
            data = data.to(DEVICE, non_blocking=True)
            target = target.to(DEVICE, non_blocking=True)
            loss = clscrit(model(data)["out"], target)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    print("************* ", end="")
    test_score = get_score(model, test_loader, DEVICE)
    test_score_list.append(test_score)
    test_recons_loss = get_recons_loss(model, test_loader, DEVICE)
    for recons_key in recons_keys:
        test_recons_loss_list[recons_key].append(test_recons_loss[recons_key])
    if train_record:
        train_score = get_score(model, train_loader, DEVICE)
        train_score_list.append(train_score)
        train_recons_loss = get_recons_loss(model, train_loader, DEVICE)
        for recons_key in recons_keys:
            train_recons_loss_list[recons_key].append(train_recons_loss[recons_key])
        print(
            f"Epoch [{epoch + 1}/{num_epochs}] "
            f"train_score={train_score:.4f}, test_score={test_score:.4f}, "
            f"train_recons_loss={train_recons_loss}, test_recons_loss={test_recons_loss}"
        )
    else:
        print( f"Epoch [{epoch + 1}/{num_epochs}] test_score={test_score:.4f}, "
               f"test_recons_loss={test_recons_loss}")
    results = {
        "model": model.name,
        "lr": lr,
        "num_epochs": num_epochs,
        "recons_keys": recons_keys,
        "train_score_list": train_score_list,
        "test_score_list": test_score_list,
        "train_recons_loss_list": train_recons_loss_list,
        "test_recons_loss_list": test_recons_loss_list,
    }

    ckpt_path = Path(f"/data/dn/FRTP_revision1/imagecls/logs_stage/{model.name}_{lr}lr_{num_epochs}e.pth")
    ckpt_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, ckpt_path)
    print(f"saved model ckpt to {ckpt_path}")

    save_path = Path(f"/data/dn/FRTP_revision1/imagecls/logs_stage/{model.name}_{lr}lr_{num_epochs}e.pt")
    save_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(results, save_path)
    print(f"saved results to {save_path}")
    print("train_score_list =", train_score_list)
    print("test_score_list =", test_score_list)
    print("train_recons_loss_list =", train_recons_loss_list)
    print("test_recons_loss_list =", test_recons_loss_list)
    return results


37 16


In [ ]:
MODEL_NAME = "simplecnnv2"
MODELPATH = None
LR = 1e-3
EPOCHS = 30
EVERY_EPOCH = 10
TRAIN_RECORD = True

model = STAGE_RECONS_MODEL_BUILDERS[MODEL_NAME]().to(DEVICE)
if MODELPATH is not None:
    ckpt = torch.load(MODELPATH, map_location=DEVICE, weights_only=True)
    if isinstance(ckpt, dict) and "state_dict" in ckpt:
        ckpt = ckpt["state_dict"]
    model.load_state_dict(ckpt)

stage_recons_results = train_stage_recons(
    model,
    lr=LR,
    num_epochs=EPOCHS,
    train_record=TRAIN_RECORD,
    every_epoch=EVERY_EPOCH,
)
train_score_list = stage_recons_results["train_score_list"]
test_score_list = stage_recons_results["test_score_list"]
train_recons_loss_list = stage_recons_results["train_recons_loss_list"]
test_recons_loss_list = stage_recons_results["test_recons_loss_list"]


************* [DEBUG fft] spectral fallback used for 20/36 frequency systems
[DEBUG fft] spectral fallback used for 204/256 frequency systems
[DEBUG fft] spectral fallback used for 1686/1764 frequency systems
[DEBUG fft] spectral fallback used for 9998/10000 frequency systems
[DEBUG fft] spectral fallback used for 20/36 frequency systems
[DEBUG fft] spectral fallback used for 212/256 frequency systems
[DEBUG fft] spectral fallback used for 1704/1764 frequency systems
[DEBUG fft] spectral fallback used for 10000/10000 frequency systems
[DEBUG fft] spectral fallback used for 20/36 frequency systems
[DEBUG fft] spectral fallback used for 212/256 frequency systems
[DEBUG fft] spectral fallback used for 1702/1764 frequency systems
